# New Workflow

In [100]:
# %load ../src/ingestion/tiingo_client.py
import json
import os

import requests
from jsonschema import validate

# Load the environment variables
TIINGO_API_KEY = os.getenv("TIINGO_API_KEY")

tiingo_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "date": {"type": "string", "format": "date-time"},
            "close": {"type": "number"},
            "high": {"type": "number"},
            "low": {"type": "number"},
            "open": {"type": "number"},
            "volume": {"type": "number"},
            "adjClose": {"type": "number"},
            "adjHigh": {"type": "number"},
            "adjLow": {"type": "number"},
            "adjOpen": {"type": "number"},
            "adjVolume": {"type": "number"},
            "divCash": {"type": "number"},
            "splitFactor": {"type": "number"},
        },
    },
}


def ticker_session_request(tickers, start_date, end_date):
    base_url = "https://api.tiingo.com/tiingo/daily/"
    request_dct = {}
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Token {TIINGO_API_KEY}",
    }
    params = {"startDate": start_date, "endDate": end_date}
    with requests.Session() as s:
        s.headers.update(headers)
        for ticker in tickers:
            request_info = {}
            try:
                url = base_url + f"{ticker}/prices"
                r = s.get(url=url, params=params, timeout=100)
                # Will raise an error if the status is not successful
                r.raise_for_status()
                # validate Json
                data = r.json()
                # Will raise an error if the schema does not match
                validate(instance=data, schema=tiingo_schema)
                ## Create request info ##
                request_info["ticker"] = ticker
                request_info["url"] = url
                request_info["status"] = r.status_code
                request_info.update(params)
                #########################
                request_dct[ticker] = {
                    "metadata": "<INSERT METADATA>",
                    "request_info": request_info,
                    "data": data,
                }
            except Exception as e:
                print(f"Error: {e}")
    return request_dct


In [101]:
start_date = '2026-01-01'
end_date = '2026-01-05'
tickers = ['JPM', 'MSFT']
out_dct = ticker_session_request(
    tickers=tickers,
    start_date=start_date,
    end_date=end_date
    )

In [104]:
out_dct.keys()

dict_keys(['JPM', 'MSFT'])

In [113]:
out_dct

{'JPM': {'metadata': '<INSERT METADATA>',
  'request_info': {'ticker': 'JPM',
   'url': 'https://api.tiingo.com/tiingo/daily/JPM/prices',
   'status': 200,
   'startDate': '2026-01-01',
   'endDate': '2026-01-05'},
  'data': [{'date': '2026-01-02T00:00:00.000Z',
    'close': 325.48,
    'high': 325.73,
    'low': 320.74,
    'open': 322.5,
    'volume': 8054040,
    'adjClose': 324.0274398262,
    'adjHigh': 324.2763241201,
    'adjLow': 319.3085936152,
    'adjOpen': 321.0607390438,
    'adjVolume': 8054040,
    'divCash': 0.0,
    'splitFactor': 1.0},
   {'date': '2026-01-05T00:00:00.000Z',
    'close': 334.04,
    'high': 337.25,
    'low': 325.19,
    'open': 325.5,
    'volume': 10752627,
    'adjClose': 332.5492380471,
    'adjHigh': 335.7449123799,
    'adjLow': 323.7387340454,
    'adjOpen': 324.0473505698,
    'adjVolume': 10752627,
    'divCash': 0.0,
    'splitFactor': 1.0}]},
 'MSFT': {'metadata': '<INSERT METADATA>',
  'request_info': {'ticker': 'MSFT',
   'url': 'https://

# Load data from Tiingo

In [7]:
import os
import json
import requests
from dotenv import load_dotenv

# Load the environment variables
load_dotenv()
TIINGO_API_KEY=os.getenv('TIINGO_API_KEY')

# This is the header format that is needed for TIINGO
headers = {
        'Content-Type': 'application/json',
        'Authorization' : f'Token {TIINGO_API_KEY}'
        }
# Submit the request
requestResponse = requests.get("https://api.tiingo.com/api/test/",
                                    headers=headers)
print(requestResponse.json())

{'message': 'You successfully sent a request'}


API Endpoints
- Meta Data
```https://api.tiingo.com/tiingo/daily/<ticker>```
  - {'ticker': 'JPM',
 'name': 'JPMorgan Chase & Company',
 'description': 'JPMorgan Chase & Co. is a leading financial services firm based in the United States of America ("U.S."), with operations worldwide. JPMorganChase had $4.4 trillion in assets and $362 billion in stockholders\' equity as of December 31, 2025. The Firm is a leader in investment banking, financial services for consumers and small businesses, commercial banking, financial transaction processing and asset management. Under the J.P. Morgan and Chase brands, the Firm serves millions of customers in the U.S., and many of the world\'s most prominent corporate, institutional and government clients globally.',
 'startDate': '1983-12-30',
 'endDate': '2026-03-11',
 'exchangeCode': 'NYSE'}
- Latest Price
```https://api.tiingo.com/tiingo/daily/<ticker>/prices```

  - [{'adjClose': 287.52,
  'adjHigh': 290.4825,
  'adjLow': 284.86,
  'adjOpen': 288.81,
  'adjVolume': 10204994,
  'close': 287.52,
  'date': '2026-03-11T00:00:00+00:00',
  'divCash': 0.0,
  'high': 290.4825,
  'low': 284.86,
  'open': 288.81,
  'splitFactor': 1.0,
  'volume': 10204994}]
- Historical Prices
```https://api.tiingo.com/tiingo/daily/<ticker>/prices?startDate=2012-1-1&endDate=2016-1-1 ```
  - [{'date': '2025-10-01T00:00:00.000Z',
  'close': 310.71,
  'high': 314.59,
  'low': 307.41,
  'open': 313.97,
  'volume': 9235211,
  'adjClose': 307.8299057865,
  'adjHigh': 311.6739405277,
  'adjLow': 304.5604947952,
  'adjOpen': 311.0596875536,
  'adjVolume': 9235211,
  'divCash': 0.0,
  'splitFactor': 1.0},
 {'date': '2025-10-02T00:00:00.000Z',
  'close': 307.55,
  'high': 310.56,
  'low': 306.14,
  'open': 310.0,
  'volume': 7599973,
  'adjClose': 304.6991970797,
  'adjHigh': 307.681296196,
  'adjLow': 303.3022669289,
  'adjOpen': 307.1264870581,
  'adjVolume': 7599973,
  'divCash': 0.0,
...}]


In [ ]:
start_date="2025-10-1"
end_date="2026-1-1"
ticker = 'JPM'
def submit_tiingo_request(tiingo_url):
    try:
        headers = {
            'Content-Type': 'application/json',
            'Authorization' : f'Token {TIINGO_API_KEY}'
            }
        tiingo_resp = requests.get(url=tiingo_url, headers=headers)
        return tiingo_resp.json()
    except Exception as e:
        # Catch other unexpected exceptions and structure a generic JSON message
        error_message = {
            "status": e.status_code,
            "message": "An unexpected error occurred",
            "details": str(e)
        }
        return error_message
    
meta_data_url = f'https://api.tiingo.com/tiingo/daily/{ticker}'
latest_price_url = f'https://api.tiingo.com/tiingo/daily/{ticker}/prices'
historical_url = f'https://api.tiingo.com/tiingo/daily/{ticker}/prices?startDate={start_date}&endDate={end_date}'
resp = submit_tiingo_request(historical_url)
resp

<Response [200]>

In [ ]:
#resp.json()

[{'date': '2025-10-01T00:00:00.000Z',
  'close': 310.71,
  'high': 314.59,
  'low': 307.41,
  'open': 313.97,
  'volume': 9235211,
  'adjClose': 307.8299057865,
  'adjHigh': 311.6739405277,
  'adjLow': 304.5604947952,
  'adjOpen': 311.0596875536,
  'adjVolume': 9235211,
  'divCash': 0.0,
  'splitFactor': 1.0},
 {'date': '2025-10-02T00:00:00.000Z',
  'close': 307.55,
  'high': 310.56,
  'low': 306.14,
  'open': 310.0,
  'volume': 7599973,
  'adjClose': 304.6991970797,
  'adjHigh': 307.681296196,
  'adjLow': 303.3022669289,
  'adjOpen': 307.1264870581,
  'adjVolume': 7599973,
  'divCash': 0.0,
  'splitFactor': 1.0},
 {'date': '2025-10-03T00:00:00.000Z',
  'close': 310.03,
  'high': 311.66,
  'low': 308.21,
  'open': 308.51,
  'volume': 6029854,
  'adjClose': 307.1562089762,
  'adjHigh': 308.7710998597,
  'adjLow': 305.3530792779,
  'adjOpen': 305.650298459,
  'adjVolume': 6029854,
  'divCash': 0.0,
  'splitFactor': 1.0},
 {'date': '2025-10-06T00:00:00.000Z',
  'close': 309.18,
  'high': 

## Change this to a session instead for multiple requests

In [ ]:
from jsonschema import validate

# Create request schema
tiingo_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "date": {"type": "string","format":"date-time"},
            "close": {"type": "number"},
            "high": {"type": "number"},
            "low": {"type": "number"},
            "open": {"type": "number"},
            "volume": {"type": "number"},
            "adjClose": {"type": "number"},
            "adjHigh": {"type": "number"},
            "adjLow": {"type": "number"},
            "adjOpen": {"type": "number"},
            "adjVolume": {"type": "number"},
            "divCash": {"type": "number"},
            "splitFactor": {"type": "number"}
        }
    }
}
validate(instance=resp.json(), schema=tiingo_schema)
# Will throw an erro if the response is not the same

In [ ]:
start_date="2025-10-1"
end_date="2026-1-1"

base_url = 'https://api.tiingo.com/tiingo/daily/'
request_dct = {}
headers = {
            'Content-Type': 'application/json',
            'Authorization' : f'Token {TIINGO_API_KEY}'
            }
tickers = ['JPM', 'AAPL']
params = {'startDate':start_date, 'endDate':end_date}
with requests.Session() as s:
    try:
        s.headers.update(headers)
        for ticker in tickers:
            url = base_url + f"{ticker}/prices"
            r = s.get(url=url, params=params, timeout=10000)
            r.raise_for_status()
            # validate Json
            validate(instance=r.json(), schema=tiingo_schema)
            request_dct[ticker] = {'metadata':"<INSERT METADATA>",
                                   'request_info':"<INSERT REQUEST INFO>",
                                   'data': r.json()}
    except Exception as e:
        print(f"Error: {e}")


In [90]:
request_dct['JPM']['data'][:3]

[{'date': '2025-10-01T00:00:00.000Z',
  'close': 255.45,
  'high': 258.79,
  'low': 254.93,
  'open': 255.04,
  'volume': 48713940,
  'adjClose': 254.9623394965,
  'adjHigh': 258.2959633521,
  'adjLow': 254.4433321896,
  'adjOpen': 254.5531221968,
  'adjVolume': 48713940,
  'divCash': 0.0,
  'splitFactor': 1.0},
 {'date': '2025-10-02T00:00:00.000Z',
  'close': 257.13,
  'high': 258.18,
  'low': 254.15,
  'open': 256.575,
  'volume': 42630239,
  'adjClose': 256.639132334,
  'adjHigh': 257.6871278575,
  'adjLow': 253.6648212293,
  'adjOpen': 256.085191843,
  'adjVolume': 42630239,
  'divCash': 0.0,
  'splitFactor': 1.0},
 {'date': '2025-10-03T00:00:00.000Z',
  'close': 258.02,
  'high': 259.24,
  'low': 253.95,
  'open': 254.665,
  'volume': 49155614,
  'adjClose': 257.5274333015,
  'adjHigh': 258.7451042907,
  'adjLow': 253.4652030344,
  'adjOpen': 254.1788380813,
  'adjVolume': 49155614,
  'divCash': 0.0,
  'splitFactor': 1.0}]

In [92]:
for k,v in request_dct['JPM']['data'][0].items():
    print(f"Key: {k} | val_dtype: {type(v)}")

Key: date | val_dtype: <class 'str'>
Key: close | val_dtype: <class 'float'>
Key: high | val_dtype: <class 'float'>
Key: low | val_dtype: <class 'float'>
Key: open | val_dtype: <class 'float'>
Key: volume | val_dtype: <class 'int'>
Key: adjClose | val_dtype: <class 'float'>
Key: adjHigh | val_dtype: <class 'float'>
Key: adjLow | val_dtype: <class 'float'>
Key: adjOpen | val_dtype: <class 'float'>
Key: adjVolume | val_dtype: <class 'int'>
Key: divCash | val_dtype: <class 'float'>
Key: splitFactor | val_dtype: <class 'float'>


In [ ]:
def ticker_session_request(tickers, start_date, end_date):
    base_url = 'https://api.tiingo.com/tiingo/daily/'
    request_dct = {}
    headers = {
                'Content-Type': 'application/json',
                'Authorization' : f'Token {TIINGO_API_KEY}'
                }
    params = {'startDate':start_date, 'endDate':end_date}
    with requests.Session() as s:
        s.headers.update(headers)
        for ticker in tickers:
            request_info = {}
            try:
                url = base_url + f"{ticker}/prices"
                r = s.get(url=url, params=params, timeout=100)
                r.raise_for_status()
                # validate Json
                data = r.json()
                validate(instance=data, schema=tiingo_schema)
                ## Create request info ##
                request_info['ticker'] = ticker
                request_info['url'] = url
                request_info['status'] = r.status_code
                request_info.update(params)

                request_dct[ticker] = {'metadata':"<INSERT METADATA>",
                                    'request_info':request_info,
                                    'data': data}
            except Exception as e:
                print(f"Error: {e}")
    return request_dct

dct = ticker_session_request(['JPM', "GOOG"],start_date="2026-1-1", end_date="2026-2-1")
dct

{'JPM': {'metadata': '<INSERT METADATA>',
  'request_info': {'ticker': 'JPM',
   'url': 'https://api.tiingo.com/tiingo/daily/JPM/prices',
   'status': 200,
   'startDate': '2026-1-1',
   'endDate': '2026-2-1'},
  'data': [{'date': '2026-01-02T00:00:00.000Z',
    'close': 325.48,
    'high': 325.73,
    'low': 320.74,
    'open': 322.5,
    'volume': 8054040,
    'adjClose': 324.0274398262,
    'adjHigh': 324.2763241201,
    'adjLow': 319.3085936152,
    'adjOpen': 321.0607390438,
    'adjVolume': 8054040,
    'divCash': 0.0,
    'splitFactor': 1.0},
   {'date': '2026-01-05T00:00:00.000Z',
    'close': 334.04,
    'high': 337.25,
    'low': 325.19,
    'open': 325.5,
    'volume': 10752627,
    'adjClose': 332.5492380471,
    'adjHigh': 335.7449123799,
    'adjLow': 323.7387340454,
    'adjOpen': 324.0473505698,
    'adjVolume': 10752627,
    'divCash': 0.0,
    'splitFactor': 1.0},
   {'date': '2026-01-06T00:00:00.000Z',
    'close': 334.61,
    'high': 335.87,
    'low': 330.65,
    '

# Convert JSON response to pandas dataframe

In [120]:
# %load ../src/processing/tiingo_to_df.py
import logging

import pandas as pd

logger = logging.getLogger(__name__)


def convert_tiingo_payload(tiingo_dct, format="long"):
    df = pd.DataFrame()
    for ticker, payload in tiingo_dct.items():
        ticker_df = pd.DataFrame(payload.get("data"))
        ticker_df["ticker"] = ticker
        # Convert date to date:
        ticker_df["date"] = pd.to_datetime(ticker_df["date"])
        df = pd.concat([df, ticker_df])
    df.set_index(["ticker", "date"], inplace=True)
    return df


In [121]:
df = convert_tiingo_payload(out_dct)
df

close    high     low     open    volume  \
ticker date                                                                   
JPM    2026-01-02 00:00:00+00:00  325.48  325.73  320.74  322.500   8054040   
       2026-01-05 00:00:00+00:00  334.04  337.25  325.19  325.500  10752627   
MSFT   2026-01-02 00:00:00+00:00  472.94  484.66  470.16  484.385  25571567   
       2026-01-05 00:00:00+00:00  472.85  476.07  469.50  474.055  25250260   

                                    adjClose     adjHigh      adjLow  \
ticker date                                                            
JPM    2026-01-02 00:00:00+00:00  324.027440  324.276324  319.308594   
       2026-01-05 00:00:00+00:00  332.549238  335.744912  323.738734   
MSFT   2026-01-02 00:00:00+00:00  471.862364  483.555659  469.088699   
       2026-01-05 00:00:00+00:00  471.772569  474.985232  468.430203   

                                     adjOpen  adjVolume  divCash  splitFactor  
ticker date                                                                    
JPM    2026-01-02 00:00:00+00:00  321.060739    8054040      0.0          1.0  
       2026-01-05 00:00:00+00:00  324.047351   10752627      0.0          1.0  
MSFT   2026-01-02 00:00:00+00:00  483.281286   25571567      0.0          1.0  
       2026-01-05 00:00:00+00:00  472.974824   25250260      0.0          1.0

In [122]:
df.dtypes

close          float64
high           float64
low            float64
open           float64
volume           int64
adjClose       float64
adjHigh        float64
adjLow         float64
adjOpen        float64
adjVolume        int64
divCash        float64
splitFactor    float64
dtype: object